In [7]:
!pip install qiskit scikit-learn rdkit-pypi

ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi


In [6]:
!pip install qiskit
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

  Using cached qiskit-2.4.1-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 6.7 MB/s eta 0:00:00


In [4]:
# Example normalized features (4 features per molecule)
X = np.array([
    [0.35,0.72,0.21,0.66],
    [0.5,0.6,0.3,0.2],
    [0.1,0.2,0.9,0.8],
    [0.9,0.1,0.4,0.3],
    [0.2,0.8,0.5,0.7],
    [0.6,0.3,0.2,0.9]
])

# Labels: 1 = toxic, 0 = non-toxic
y = np.array([1,0,1,0,1,0])

In [3]:
def quantum_circuit(features, theta):
    qc = QuantumCircuit(4)

    # Encoding layer
    for i in range(4):
        qc.ry(features[i], i)

    # Entanglement layer
    qc.cx(0,1)
    qc.cx(1,2)
    qc.cx(2,3)

    # Variational layer
    for i in range(4):
        qc.ry(theta[i], i)

    return qc

In [8]:
def get_probabilities(features, theta):
    qc = quantum_circuit(features, theta)
    state = Statevector.from_instruction(qc)
    return state.probabilities()

In [9]:
def quantum_features(features, theta):
    probs = get_probabilities(features, theta)

    return np.array([
        probs[0],         # |0000>
        probs[-1],        # |1111>
        np.mean(probs),
        np.max(probs)
    ])

In [10]:
theta = np.random.rand(4)

In [11]:
X_q = np.array([quantum_features(x, theta) for x in X])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X_q, y, test_size=0.3, random_state=42)

In [13]:
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [14]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.5


In [15]:
def predict_toxicity(x_new):
    q_feat = quantum_features(x_new, theta)
    pred = model.predict([q_feat])
    return "Toxic" if pred[0] == 1 else "Non-Toxic"

In [16]:
sample = [0.4, 0.7, 0.2, 0.6]
print("Prediction:", predict_toxicity(sample))

Prediction: Non-Toxic
